# Lifespan -- Kaggle pilot runner

A thin caller. It checks out this repo at **one commit** (a parameter, below),
installs nothing (Kaggle's image already has torch, numpy, tokenizers and
matplotlib), and calls `python -m training.train ...` -- the frozen CLI
contract in `AGENTS.md`. No training logic lives in a cell.

**Rules this notebook holds itself to (`AGENTS.md`, `CLAUDE.md`):**
- Training data and exams are two separate **private** Kaggle datasets,
  mounted read-only and passed as `--train-dir` / `--exam-dir`. Exam text is
  never read, never printed, and never uploaded as a public dataset.
- Kaggle credentials (`KAGGLE_USERNAME`/`KAGGLE_KEY`) come from the
  environment or Kaggle's own Secrets add-on. They are never written in this
  notebook and never printed.
- **Commit this notebook with outputs cleared.** `Kernel/Restart & Clear
  Output` before every commit; `notebooks/check_outputs_cleared.py` (checked
  in beside this file) asserts it.
- The session's GPU (T4 or P100) is detected, never assumed, and recorded in
  the result folder's `timings.json`. A run's arms must be compared on the
  same GPU model -- resuming on a different model is refused below.
- A Kaggle session dies at 12 hours or whenever it likes. After every phase
  the partial result folder (and the shared init / phase-0 checkpoint) is
  pushed to a Kaggle *dataset* version, so the next session can pull the
  latest version and resume with `--resume`.

In [ ]:
# --- Parameters: edit these per session (or override via Kaggle's notebook ---
# --- "Variables", or papermill). None of these is a credential.            ---

COMMIT = "REPLACE_WITH_THE_EXACT_COMMIT_SHA"  # the run's provenance; never "HEAD"
REPO_GIT_URL = "REPLACE_WITH_THE_REPO_REMOTE_URL"  # auth via env/Kaggle secret in the git credential helper, never embedded here

ARM = "A"          # one of: phase0, A, B, C, D, D-nr, E
SEED = 0           # one of: 0, 1, 2
PILOT = True       # sizes only (AGENTS.md, frozen): never a hyperparameter
RESUME = False     # True to continue a session this notebook (or an earlier one) started

# Two SEPARATE private Kaggle datasets, mounted read-only under /kaggle/input.
# Exam text must never be a public dataset and never printed by any cell below.
TRAIN_DATASET_DIR = "/kaggle/input/lifespan-train-data"
EXAM_DATASET_DIR = "/kaggle/input/lifespan-exam-data"

# A Kaggle dataset THIS notebook owns and pushes new versions to after every
# phase; on RESUME its latest version must be attached as a notebook input
# too, mounted at /kaggle/input/<the part after the last '/'>.
CHECKPOINT_DATASET_SLUG = "REPLACE_WITH_owner-username/lifespan-checkpoints"

# Exam manifest, phase subset and experiment id (AGENTS.md, Amendment 3).
# None = the main experiment's defaults: exams/manifest.json in the checkout,
# all seven phases, training.config.EXPERIMENT_ID ("lifespan-main").
# The 0/3/6 pre-pilot (docs/DECISIONS.md, 2026-09-25 and 2026-09-30) is its own
# throwaway experiment and sets all three:
#   MANIFEST_PATH = f"{EXAM_DATASET_DIR}/manifest.json"
#   PHASES = "0,3,6"
#   EXPERIMENT_ID = "lifespan-prepilot"
MANIFEST_PATH = None
PHASES = None
EXPERIMENT_ID = None

# Rows per forward/backward (--micro-batch). Gradients accumulate to the full
# 32-sequence batch before each optimiser step: memory only, never a
# hyperparameter. A T4 (15 GB) runs out of memory on the whole batch; 8 fits.
MICRO_BATCH = 8

In [ ]:
# --- Checkout: this repo, at exactly COMMIT. No shell script -- plain subprocess calls. ---
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Lifespan")

import base64
import os

# Credentials from Kaggle's Secrets add-on (Add-ons > Secrets), loaded into
# this process's environment only: never printed, never written to disk.
#   GITHUB_TOKEN     optional; only needed if the repo is private (fine-grained PAT)
#   KAGGLE_USERNAME  / KAGGLE_KEY  for the checkpoint-dataset pushes
try:
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    for _name in ("GITHUB_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        if not os.environ.get(_name):
            try:
                os.environ[_name] = _secrets.get_secret(_name)
            except Exception:
                pass
except ImportError:
    pass  # not on Kaggle: rely on the environment

# The token travels as an HTTP header for these git calls only, so it is never
# part of a URL, the clone's .git/config, or any output.
_git_auth = []
if os.environ.get("GITHUB_TOKEN"):
    _basic = base64.b64encode(f"x-access-token:{os.environ['GITHUB_TOKEN']}".encode()).decode()
    _git_auth = ["-c", f"http.extraHeader=Authorization: Basic {_basic}"]

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", *_git_auth, "clone", REPO_GIT_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", *_git_auth, "-C", str(REPO_DIR), "fetch", "--all"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", COMMIT], check=True)

status = subprocess.run(
    ["git", "-C", str(REPO_DIR), "status", "--porcelain"], capture_output=True, text=True, check=True
)
assert not status.stdout.strip(), (
    "the checked-out tree is dirty right after checkout -- something in the "
    "image or the clone modified it; refusing to produce a result from it"
)
print(f"checked out {COMMIT} clean")

In [ ]:
# --- Which GPU did this session get? Detected, never assumed (PLAN.md). ---
import torch

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"session GPU: {GPU_NAME}")

In [ ]:
# --- Resolve the local run directory; pull the latest checkpoint dataset version if resuming. ---
import json
import shutil

RUN_TAG = f"{ARM}_s{SEED}"
LOCAL_OUT = Path("/kaggle/working/out") / RUN_TAG
LOCAL_PHASE0_DIR = Path("/kaggle/working/_phase0")

if RESUME:
    mounted = Path("/kaggle/input") / CHECKPOINT_DATASET_SLUG.split("/")[-1]
    assert mounted.exists(), (
        f"RESUME=True but {mounted} is not mounted -- attach the checkpoint "
        "dataset's latest version as a notebook input first"
    )
    mounted_run = mounted / "out" / RUN_TAG
    assert mounted_run.exists(), f"{mounted_run} not found in the checkpoint dataset -- nothing to resume"

    prev_timings = mounted_run / "timings.json"
    if prev_timings.exists():
        prev_gpu = json.loads(prev_timings.read_text(encoding="utf-8")).get("gpu_name")
        assert prev_gpu in (None, GPU_NAME), (
            f"this session got {GPU_NAME!r} but the run so far used {prev_gpu!r} -- "
            "a run's arms must be compared on the same GPU model; refusing to mix"
        )

    if LOCAL_OUT.exists():
        shutil.rmtree(LOCAL_OUT)
    shutil.copytree(mounted_run, LOCAL_OUT)
    if (mounted / "_phase0").exists():
        LOCAL_PHASE0_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(mounted / "_phase0", LOCAL_PHASE0_DIR, dirs_exist_ok=True)
    print(f"resuming {RUN_TAG} from {mounted_run} (GPU {GPU_NAME})")
else:
    LOCAL_OUT.mkdir(parents=True, exist_ok=True)
    print(f"starting {RUN_TAG} fresh at {LOCAL_OUT} (GPU {GPU_NAME})")

In [ ]:
# --- A background watcher: push a new Kaggle dataset version after every completed phase. ---
# This is the checkpoint half of surviving a dead session -- the training
# subprocess below already rewrites `state.pt`/`matrix.json`/`timings.json`
# after every phase (training/checkpoint.py, training/runrecord.py); this
# watcher's only job is getting that local folder off the Kaggle VM before it
# disappears. It reads only JSON (never `training/`'s internals), same as
# `runbook/resume_roundtrip.py`'s local proof of this exact mechanism.
import threading
import time


def _phases_done(out_dir: Path) -> int:
    path = out_dir / "timings.json"
    if not path.exists():
        return 0
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return 0
    return sum(1 for row in data.get("per_phase", []) if row.get("eval_s", 0) > 0)


def _push_checkpoint_dataset(out_dir: Path, phase0_dir: Path, slug: str, message: str) -> None:
    staging = Path("/kaggle/working/_dataset_push")
    if staging.exists():
        shutil.rmtree(staging)
    (staging / "out").mkdir(parents=True)
    shutil.copytree(out_dir, staging / "out" / out_dir.name)
    if phase0_dir.exists():
        shutil.copytree(phase0_dir, staging / "_phase0")
    (staging / "dataset-metadata.json").write_text(
        json.dumps({"title": slug.split("/")[-1], "id": slug, "licenses": [{"name": "CC0-1.0"}]}),
        encoding="utf-8",
    )
    # kaggle CLI auth is KAGGLE_USERNAME/KAGGLE_KEY from the environment or the
    # Kaggle Secrets add-on -- never written here, never printed.
    pushed = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(staging), "-m", message, "-r", "tar"],
        check=False,
    )
    if pushed.returncode != 0:
        # The first push of a run: the dataset does not exist yet, so create it
        # (private by default). A failure here is reported, never silent.
        created = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(staging), "-r", "tar"],
            check=False,
        )
        if created.returncode != 0:
            print(f"WARNING: checkpoint push failed ({message}); check KAGGLE_USERNAME/KAGGLE_KEY and {slug}")


def _watch(out_dir: Path, phase0_dir: Path, slug: str, stop_event: threading.Event) -> None:
    seen = 0
    while not stop_event.wait(timeout=15):
        n = _phases_done(out_dir)
        if n > seen:
            _push_checkpoint_dataset(out_dir, phase0_dir, slug, f"phase {n - 1} complete")
            seen = n
    _push_checkpoint_dataset(out_dir, phase0_dir, slug, "session end")


_stop_event = threading.Event()
_watcher = threading.Thread(
    target=_watch, args=(LOCAL_OUT, LOCAL_PHASE0_DIR, CHECKPOINT_DATASET_SLUG, _stop_event), daemon=True
)
_watcher.start()
print("checkpoint watcher started: pushes a dataset version after every completed phase")

In [ ]:
# --- The one call that matters: the frozen CLI contract, nothing else. ---
import sys

argv = [
    sys.executable,
    "-m",
    "training.train",
    "--arm",
    ARM,
    "--seed",
    str(SEED),
    "--train-dir",
    TRAIN_DATASET_DIR,
    "--exam-dir",
    EXAM_DATASET_DIR,
    "--out",
    str(LOCAL_OUT),
    "--phase0-dir",
    str(LOCAL_PHASE0_DIR),
]
if MANIFEST_PATH:
    argv += ["--manifest", str(MANIFEST_PATH)]
if PHASES:
    argv += ["--phases", str(PHASES)]
if EXPERIMENT_ID:
    argv += ["--experiment-id", str(EXPERIMENT_ID)]
if MICRO_BATCH:
    argv += ["--micro-batch", str(MICRO_BATCH)]
if PILOT:
    argv.append("--pilot")
if RESUME:
    argv.append("--resume")

print("running: python -m training.train " + " ".join(argv[3:]))
try:
    # Less fragmentation between the eval and training allocations.
    _env = {**os.environ, "PYTORCH_ALLOC_CONF": "expandable_segments:True"}
    proc = subprocess.run(argv, cwd=str(REPO_DIR), env=_env)
finally:
    _stop_event.set()
    _watcher.join(timeout=120)

assert proc.returncode == 0, f"training.train exited {proc.returncode}"
print(f"done: {LOCAL_OUT}")

## Before committing this notebook

1. `Kernel > Restart & Clear Output` (or Kaggle's "Clear outputs" action).
2. Run `python notebooks/check_outputs_cleared.py notebooks/lifespan_kaggle.ipynb`
   locally and confirm it prints `clean`.
3. Never commit a run where this notebook's *own* output cells show exam
   text, a token, or a Kaggle key -- if any cell above ever prints one, that
   is a bug in the cell, not something to hide by clearing output after the
   fact.